In [ ]:
# import os
# import time
# import numpy as np
# import pandas as pd
# import yfinance as yf
# import kagglehub
# from grad_fw import FWHomotopySolver
# from grad_fw.benchmarks.GreedySolver import GreedySolver

# # PATHS
# REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # examples/market -> repo root
# DATA_DIR = os.path.join(REPO_ROOT, "data", "market")
# os.makedirs(DATA_DIR, exist_ok=True)

# KAGGLE_DATASET = "andrewmvd/sp-500-stocks"
# PRICES_CACHE = os.path.join(DATA_DIR, "sp500_prices.csv")
# MAX_AGE_DAYS = 30

# print(f"REPO_ROOT: {REPO_ROOT}")
# print(f"DATA_DIR:  {DATA_DIR}")


# # Kaggle Data
# def load_kaggle_data(force_update=False):
#     """load_kaggle_data :Check if data is outdated, update if needed

#     Args:
#         force_update (bool, optional): _description_. Defaults to False.

#     Returns:
#         _type_: _description_
#     """
#     companies_path = os.path.join(DATA_DIR, "sp500_companies.csv")
#     if not force_update and os.path.exists(companies_path):
#         age = (time.time() - os.path.getmtime(companies_path)) / 86400
#         if age < MAX_AGE_DAYS:
#             print(f"Using cached Kaggle data ({age:.0f} days old).")
#         else:
#             print(f"Kaggle cache is {age:.0f} days old, re-downloading...")
#             kagglehub.dataset_download(KAGGLE_DATASET, output_dir=DATA_DIR)
#     else:
#         print("Downloading Kaggle dataset...")
#         kagglehub.dataset_download(KAGGLE_DATASET, output_dir=DATA_DIR)

#     return {
#         name: pd.read_csv(os.path.join(DATA_DIR, f"sp500_{name}.csv"))
#         for name in ["companies", "index"]
#     }


# # Price Data (cached)
# def load_prices(tickers, start="2015-01-01", end="2024-12-31", force_update=False):
#     if not force_update and os.path.exists(PRICES_CACHE):
#         age = (time.time() - os.path.getmtime(PRICES_CACHE)) / 86400
#         if age < MAX_AGE_DAYS:
#             print(f"Loading cached prices ({age:.0f} days old)...")
#             return pd.read_csv(PRICES_CACHE, index_col="Date", parse_dates=True)
#         print(f"Price cache is {age:.0f} days old, re-downloading...")
#     else:
#         print("Downloading prices from yfinance (this takes ~2 min)...")

#     raw = yf.download(tickers, start=start, end=end, auto_adjust=False)["Adj Close"]
#     raw = raw.dropna(axis=1, thresh=int(0.95 * len(raw)))
#     raw = raw.ffill().dropna()
#     raw.index.name = "Date"
#     raw.to_csv(PRICES_CACHE)
#     print(f"Saved {raw.shape[1]} stocks x {raw.shape[0]} days → {PRICES_CACHE}")
#     return raw


# # Return and Covariance matrix
# def build_matrices(prices):
#     X = np.log(prices / prices.shift(1)).dropna().values  # (T, p)
#     A = np.cov(X.T)
#     return X, A


/Users/nautilus/gridfw/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT: /Users/nautilus/gridfw
DATA_DIR:  /Users/nautilus/gridfw/data/market


In [ ]:
# dfs = load_kaggle_data()
# tickers = dfs["companies"]["Symbol"].str.replace(".", "-", regex=False).tolist()

# # 2. Load prices
# prices = load_prices(tickers)
# stock_names = prices.columns.tolist()
# print(f"Prices shape: {prices.shape}  (days x stocks)")


Using cached Kaggle data (10 days old).
Loading cached prices (10 days old)...
Prices shape: (2389, 464)  (days x stocks)


In [3]:
X, A = build_matrices(prices=prices)
p = A.shape[0]
print(f"X: {X.shape} | A: {A.shape}")

X: (2388, 464) | A: (464, 464)


In [4]:



# Run Solver
k = 50
total_variance = np.trace(A) 

print(f"\nRunning FW-Homotopy (k={k}, p={p})...")
solver = FWHomotopySolver(A, k, n_steps=800, n_mc_samples=100)
s = solver.solve(verbose=True)
fw_indices = np.where(s > 0.5)[0]
fw_stocks = [stock_names[i] for i in fw_indices]
print(fw_stocks)

print(f"\nRunning Greedy...")
greedy_solver = GreedySolver(A, k)
greedy_s = greedy_solver.solve()[0]
greedy_stocks = [stock_names[i] for i in greedy_s]
print(greedy_stocks)



Running FW-Homotopy (k=50, p=464)...
[FWHomotopy] p=464, k=50, steps=800, n_mc=100, alpha=0.01
['ALB', 'ALGN', 'AMD', 'ANET', 'APA', 'AXON', 'BIIB', 'BLDR', 'CCL', 'CFG', 'CZR', 'DVN', 'DXCM', 'ENPH', 'EPAM', 'EQT', 'ES', 'EXPE', 'FCX', 'FSLR', 'FTNT', 'GNRC', 'HAL', 'KEY', 'LRCX', 'MOH', 'MOS', 'MPWR', 'MTCH', 'MU', 'NCLH', 'NFLX', 'NVDA', 'OKE', 'ON', 'OXY', 'PAYC', 'PCG', 'PODD', 'SMCI', 'SW', 'TPL', 'TPR', 'TRGP', 'TSLA', 'UAL', 'VTR', 'WBD', 'WDC', 'WYNN']

Running Greedy...
['AMP', 'TEL', 'CMS', 'DVN', 'MSFT', 'CCL', 'ENPH', 'PHM', 'MPWR', 'RF', 'SMCI', 'DXCM', 'PCG', 'FRT', 'TMO', 'TRGP', 'ITW', 'TSLA', 'WYNN', 'AMD', 'ELV', 'EQT', 'PAYC', 'FCX', 'CZR', 'VLO', 'WDC', 'WBD', 'MTCH', 'APA', 'GIS', 'DAL', 'AXON', 'FSLR', 'BIIB', 'SW', 'EPAM', 'AMAT', 'MOS', 'NFLX', 'ALGN', 'TGT', 'UHS', 'UDR', 'ALB', 'INCY', 'CB', 'ANET', 'TPR', 'TPL']
